# Figure 1 

In [1]:
options(warn=-1)

In [2]:
library_load <- suppressMessages(
    
    suppressWarnings(
        
        list(
        
            # Seurat 
            library(Seurat), 
            
            # miloR
            library(miloR), 
            library(ggbeeswarm), 
            
            # GLMM
            library(lmerTest),
            library(lme4), 
            library(emmeans), 
            
            # Diversity and PERMANOVA
            library(vegan), 
            library(pairwiseAdonis), 
             
            # Data 
            library(tidyverse), 
            library(data.table), 
            library(reactable), 

            # Plotting 
            library(ComplexHeatmap), 
            library(patchwork), 
            library(cowplot), 
            library(ggrepel), 
            
            # Parallel 
            library(BiocParallel), 

            # Pyhton compatibility
            library(reticulate)

        )
    )
)

In [3]:
# Configure reticulate 
use_condaenv(condaenv="p.3.8.12-FD20220623HSCT", conda="/nobackup/peer/fdeckert/miniconda3/bin/conda", required=NULL)

In [4]:
random_seed <- 42
set.seed(random_seed)

In [5]:
# Set working directory to project root
setwd("/research/peer/fdeckert/FD20220623HSCT")

In [6]:
# Plotting Theme
source("plotting_global.R")
ggplot2::theme_set(theme_global_set(size_select=4)) # From project global source()

In [7]:
# Source scripts 
source("bin/seurat_qc.R")
source("script/figure/bin/dp_feature.R")
source("script/figure/bin/line_plot.R")

# Import Seurat object

In [8]:
so <- readRDS("data/object/pp_subset.rds")

# Marker gene expression of leiden cell type broad 

In [9]:
# Make cell type labels broad
leiden_cell_type_broad <- data.frame(

    cell_type_leiden_subset=c(
        
        
        "T['h, cm/naive']", "T['h, em/effector']", "T['h, memory/effector']", "T['reg']", "T['c, cm/naive']", "T['c, em/effector']", "T['c, memory/effector']", "T['inv']", "Cycling", 

        "NK*' CD16-'", "NK*' CD16+'",
        
        "Mono['Cl']", "Mono['Int']", "Mono['nonCl']", 
        "M*Phi", "M*Phi['Act']", "M*Phi['Inf']",
        "cDC1", "cDC2", "LC", "cDC['Mig']", "pDC", 

        "F['ret, ECM']", "F['ret, activated']", "F['ret, immune']", "F['pap, ECM']", "F['DS']", "F['DP']",
        "KC['b']", "KC['sb']",
        "vEC", "lEC", 
        "Pc", "Pc['immune']",
        "Mlc"
        
    ), 

    leiden_cell_type_broad=c(
        
        rep("T*' cells'", 9), 
        rep("NK*' cells'", 2), 
        
        rep("Monocytes", 3), 
        rep("M*Phi", 3), 
        rep("DC", 5), 

        rep("Fibroblast", 6), 
        rep("Keratinocytes", 2), 
        rep("Endothelial*' cells'", 2), 
        rep("Pericytes", 2), 
        rep("Melanocytes", 1)


    )

) %>% dplyr::mutate(leiden_cell_type_broad=factor(leiden_cell_type_broad, levels=unique(leiden_cell_type_broad)))

In [10]:
# Add labels
so <- AddMetaData(so, dplyr::left_join(so@meta.data[, c("cell_id", "cell_type_leiden_subset")], leiden_cell_type_broad, by=join_by(cell_type_leiden_subset)) %>% tibble::column_to_rownames("cell_id"))

In [11]:
# Normalize 
so <- NormalizeData(so, assay="RNA", scale.factor=1e6)

In [12]:
# Genes and categories
genes <- read.csv("script/figure/figure_1/data/figure_1_dotplot_celltype_markers_broad.csv")

In [13]:
dp_1 <- dp_feature(so, genes$gene, group_by="leiden_cell_type_broad", group_by_order=rev(levels(so$leiden_cell_type_broad)), split=genes$celltype, split_order=genes$celltype, range_max=3, scale=TRUE) + 
    scale_y_discrete(labels=base::str2expression(rev(levels(so$leiden_cell_type_broad)))) 

`summarise()` has grouped output by 'leiden_cell_type_broad'. You can override
using the `.groups` argument.
Scale for fill is already present.
Adding another scale for fill, which will replace the existing scale.


In [14]:
pdf("result/figures/figure_1/dp_feature_leiden_cell_type_broad.pdf", width=3.5, height=2.5)

dp_1

dev.off()

png 
  2

## Patient distribution within broad cell types 

In [46]:
bp_1 <- ggplot(so@meta.data, aes(x=leiden_cell_type_broad, fill=patient_id)) + 
    geom_bar(position="fill", width=0.8, color="black", size=0.1) + 
    scale_fill_manual(values=color$patient_id) + 
    scale_x_discrete(labels=base::str2expression(levels(so$leiden_cell_type_broad))) + 
    ggtitle("Patient composition") + xlab("") + ylab("Patient ID [ratio]") + 
    theme(axis.text.x=element_text(angle=90, vjust=0.5, hjust=1)) + 
    guides(fill=guide_legend(ncol=1, override.aes=list(alpha=1, size=1.5), keywidth=0.25, keyheight=0.25, default.unit="cm"))

In [47]:
pdf("result/figures/figure_1/bp_patient_id_leiden_cell_type_broad.pdf", width=3, height=2)

bp_1

dev.off()

png 
  2

# UMAP 

In [30]:
so$cell_type_leiden_subset <- factor(so$cell_type_leiden_subset, levels=names(color$cell_type_leiden_subset_r))

## Cell type leiden subset

In [ ]:
dplot_1 <- dplot(so, reduction="umap", group_by="cell_type_leiden_subset", alpha=1, pt_size=0.5, size_select=4) + 
    ggtitle("Cell type leiden") + 
    scale_color_manual(values=color$cell_type_leiden_subset_r, na.value="grey50", labels=base::str2expression(levels(so$cell_type_leiden_subset))) + 
    guides(color=guide_legend(ncol=2, override.aes=list(alpha=1, size=1.5), keywidth=0, keyheight=0.25, default.unit="cm")) + 
    theme(
    
        legend.text.align=0
    
    ) 

In [ ]:
pdf("result/figures/figure_1/umap_cell_type_leiden_subset.pdf", width=10, height=3)

dplot_1 

dev.off()

## Patient ID

In [ ]:
dplot_2 <- dplot(so, reduction="umap", group_by="patient_id", alpha=1, pt_size=0.5, size_select=4) + ggtitle("Patient ID") + 
    scale_color_manual(values=color$patient_id, na.value="grey50") + 
    guides(color=guide_legend(ncol=1, override.aes=list(alpha=1, size=1.5), keywidth=0, keyheight=0.25, default.unit="cm"))

In [ ]:
pdf("result/figures/figure_1/umap_patient_id.pdf", width=10, height=3)

dplot_2

dev.off()

## HTO demux class

In [ ]:
dplot_3 <- dplot(so, reduction="umap", group_by="hto_demux_class", alpha=1, pt_size=0.5, size_select=4) + ggtitle("Tissue") + 
    scale_color_manual(values=color$hto_demux_class, na.value="grey50") + 
    guides(color=guide_legend(ncol=1, override.aes=list(alpha=1, size=1.5), keywidth=0, keyheight=0.25, default.unit="cm"))

In [ ]:
pdf("result/figures/figure_1/umap_hto_demux_class.pdf", width=10, height=3)

dplot_3

dev.off()

## Genotype class

In [ ]:
dplot_4 <- dplot(subset(so, subset=genotype_class %in% c("Host", "Donor", "unassigned", "doublet")), reduction="umap", group_by="genotype_class", alpha=1, pt_size=0.5, size_select=4) + ggtitle("Genotype") + 
    scale_color_manual(values=color$genotype_class, na.value="grey50") + 
    guides(color=guide_legend(ncol=1, override.aes=list(alpha=1, size=1.5), keywidth=0, keyheight=0.25, default.unit="cm"))

In [ ]:
pdf("result/figures/figure_1/umap_genotype_class.pdf", width=10, height=3)

dplot_4

dev.off()

# Genotype line plots 

## Cell type ratio Host vs Donor 

In [ ]:
data <- so@meta.data %>% 

    # Select relevant entries 
    dplyr::select(domain_leiden_subset, patient_id, hto_demux_class, time_point, genotype_class) %>% droplevels() %>% 
    
    # Make domain leiden labels
    dplyr::mutate(family_domain_subset=ifelse(domain_leiden_subset=="structural", "CD45-", "CD45+")) %>% 
    dplyr::mutate(family_domain_subset=paste0(hto_demux_class, "\n(", family_domain_subset, ")")) %>%
    dplyr::mutate(family_domain_subset=factor(family_domain_subset, levels=c("Blood\n(CD45+)", "Skin\n(CD45+)", "Skin\n(CD45-)"))) %>% 

    # Remove domain leiden label 
    dplyr::select(-domain_leiden_subset) %>% 
    
    # Cell count total for hto demux domain labels 
    dplyr::group_by(family_domain_subset, patient_id, hto_demux_class, time_point) %>% 
    dplyr::mutate(cell_count_total=n()) %>%
    
    # Cell count for genotype class
    dplyr::group_by(genotype_class, .add=TRUE) %>% 
    dplyr::mutate(cell_count_gc=n()) %>%
    
    # Ratio 
    dplyr::mutate(ratio_gc=cell_count_gc/cell_count_total) %>% 
    
    # Clean up data 
    dplyr::select(-cell_count_total, -cell_count_gc) %>% 
    dplyr::distinct() %>% 

    # Fill missing genotype ratios 
    group_by(across(setdiff(group_vars(.), 'genotype_class'))) %>% 
    tidyr::complete(., genotype_class, fill=list(ratio_gc=0)) %>% 

    # Ungroup 
    dplyr::ungroup()

In [ ]:
data <- dplyr::group_by(data, time_point, family_domain_subset, genotype_class) %>% 
    dplyr::mutate(mean=mean(ratio_gc), sd=sd(ratio_gc), n=length(unique(patient_id)), sem=sd/sqrt(n), ymin=ifelse(mean-sem<0, 0, mean-sem), ymax=ifelse(mean+sem>1, 1, mean+sem)) %>%
    dplyr::select(family_domain_subset, time_point, genotype_class, mean, ymin, ymax) %>% dplyr::distinct()

In [ ]:
lp <- lp_time_point(data, title="", ylab="Genotype~ratio", filter_errorbar=c("Host", "Donor")) + facet_grid(~family_domain_subset) + scale_y_continuous(breaks=c(0.0, 0.5, 1.0), labels=c("0.0", "0.5", "1.0"))

In [ ]:
pdf("result/figures/figure_1/line_plot_genotype_class_ratio.pdf", width=3, height=2)

lp

dev.off()

## Cell type diversity 

In [ ]:
data <- so@meta.data %>%

    # Select relevant entries 
    dplyr::select(cell_type_leiden_subset, domain_leiden_subset, patient_id, hto_demux_class, time_point, genotype_class) %>% droplevels() %>% 
    
    # Make domain leiden labels
    dplyr::mutate(family_domain_subset=ifelse(domain_leiden_subset=="structural", "CD45-", "CD45+")) %>% 
    dplyr::mutate(family_domain_subset=paste0(hto_demux_class, "\n(", family_domain_subset, ")")) %>%
    dplyr::mutate(family_domain_subset=factor(family_domain_subset, levels=c("Blood\n(CD45+)", "Skin\n(CD45+)", "Skin\n(CD45-)"))) %>% 

    # Remove domain leiden label 
    dplyr::select(-domain_leiden_subset) %>% 
    
    # Cell count total for genotype class within groups
    dplyr::group_by(cell_type_leiden_subset, family_domain_subset, patient_id, time_point, genotype_class) %>%
    dplyr::summarise(cell_count_gc=n()) %>% dplyr::group_by(genotype_class, .add=TRUE) %>% # I don't know why but summarising delets the genotype_class grouping variable 

    # Filter out singeltons 
    dplyr::filter(cell_count_gc>1) %>% 
    
    # Compute richness 
    group_by(across(setdiff(group_vars(.), 'cell_type_leiden_subset'))) %>% 
    dplyr::summarise(richness=n()) %>% 

    # Fill missing genotype ratios 
    group_by(across(setdiff(group_vars(.), 'genotype_class'))) %>% 
    tidyr::complete(., genotype_class, fill=list(richness=0)) %>% 

    # Ungroup 
    dplyr::ungroup()

In [ ]:
data <- dplyr::group_by(data, family_domain_subset, time_point, genotype_class) %>% 
    dplyr::mutate(mean=mean(richness), sd=sd(richness), n=length(unique(patient_id)), sem=sd/sqrt(n), ymin=ifelse(mean-sem<0, 0, mean-sem), ymax=mean+sem) %>%
    dplyr::select(family_domain_subset, time_point, genotype_class, mean, ymin, ymax) %>% dplyr::distinct()

In [ ]:
lp <- lp_time_point(data, title="", ylab="Richness") + facet_grid(~family_domain_subset) 

In [ ]:
pdf("result/figures/figure_1/line_plot_genotype_class_richness.pdf", width=3, height=2)

lp

dev.off()

# Cell type composition 

## Cell types per patient ID 

In [ ]:
cell_count <- so@meta.data %>% 

    # Make domain leiden labels
    dplyr::mutate(family_domain_subset=ifelse(domain_leiden_subset=="structural", "CD45-", "CD45+")) %>% 
    dplyr::mutate(family_domain_subset=paste0(hto_demux_class, " (", family_domain_subset, ")")) %>%
    dplyr::mutate(family_domain_subset=factor(family_domain_subset, levels=c("Blood (CD45+)", "Skin (CD45+)", "Skin (CD45-)"))) %>% 

    # Mutate patient labels 
    dplyr::mutate(patient_id=factor(gsub("Patient_", "P", patient_id), levels=paste0("P", 1:11))) %>% 
    
    # Count cell types
    dplyr::group_by(patient_id, time_point, family_domain_subset, cell_type_leiden_subset) %>%  
    dplyr::summarise(cell_count=n()) %>% dplyr::ungroup() %>% 
    dplyr::arrange(family_domain_subset) %>% 

    # Set factor level of cell types 
    dplyr::mutate(cell_type_leiden_subset=factor(cell_type_leiden_subset, levels=names(cell_type_leiden_subset_r))) %>% 

    droplevels()

In [ ]:
label <- names(color$cell_type_leiden_subset_r)

In [ ]:
data_1 <- cell_count[cell_count$family_domain_subset=="Blood (CD45+)", ]
label_1 <- label[label %in% data_1$cell_type_leiden_subset]

bp_1 <- ggplot(data_1, aes(x=patient_id, y=log2(cell_count), fill=cell_type_leiden_subset)) + 
    geom_bar(stat="identity") + 
    ylab("log2(counts)") + xlab("") + 
    facet_grid(family_domain_subset~time_point, space="free", scales="free_x") + 
    scale_fill_manual(values=color$cell_type_leiden_subset_r, labels=base::str2expression(label_1)) + 
    theme(axis.text.x=element_text(angle=90, vjust=0.5, hjust=1)) + 
    guides(fill=guide_legend(ncol=1, override.aes=list(alpha=1, size=1.5), keywidth=0, keyheight=0.1, default.unit="cm", title=NULL))

In [ ]:
data_2 <- cell_count[cell_count$family_domain_subset=="Skin (CD45+)", ]
label_2 <- label[label %in% data_2$cell_type_leiden_subset]

bp_2 <- ggplot(data_2, aes(x=patient_id, y=log2(cell_count), fill=cell_type_leiden_subset)) + 
    geom_bar(stat="identity") + 
    ylab("log2(counts)") + xlab("") + 
    facet_grid(family_domain_subset~time_point, space="free", scales="free_x") + 
    scale_fill_manual(values=color$cell_type_leiden_subset_r, labels=base::str2expression(label_2)) + 
    theme(axis.text.x=element_text(angle=90, vjust=0.5, hjust=1)) + 
    guides(fill=guide_legend(ncol=1, override.aes=list(alpha=1, size=1.5), keywidth=0, keyheight=0.1, default.unit="cm", title=NULL))

In [ ]:
data_3 <- cell_count[cell_count$family_domain_subset=="Skin (CD45-)", ]
label_3 <- label[label %in% data_3$cell_type_leiden_subset]


bp_3 <- ggplot(data_3, aes(x=patient_id, y=log2(cell_count), fill=cell_type_leiden_subset)) + 
    geom_bar(stat="identity") + 
    ylab("log2(counts)") + xlab("") + 
    facet_grid(family_domain_subset~time_point, space="free", scales="free_x") + 
    scale_fill_manual(values=color$cell_type_leiden_subset_r, labels=base::str2expression(label_3)) + 
    theme(axis.text.x=element_text(angle=90, vjust=0.5, hjust=1)) + 
    guides(fill=guide_legend(ncol=1, override.aes=list(alpha=1, size=1.5), keywidth=0, keyheight=0.1, default.unit="cm", title=NULL))

In [ ]:
pdf("result/figures/figure_1/cell_type_per_patient_id_bar_plot.pdf", width=4, height=6)

bp_1 + bp_2 + bp_3 + patchwork::plot_layout(ncol=1)

dev.off()

# MiloR results 

In [ ]:
res_1 <- readRDS("data/object/milo/time_point/res_1.rds")[[1]]
res_2 <- readRDS("data/object/milo/time_point/res_2.rds")[[1]]
res_3 <- readRDS("data/object/milo/time_point/res_3.rds")[[3]]
res_4 <- readRDS("data/object/milo/time_point/res_4.rds")[[3]]
res_5 <- readRDS("data/object/milo/time_point/res_5.rds")[[3]]

In [ ]:
contrast <- c("Tx", "D14", "D100", "GVHD")

In [ ]:
data_1 <- lapply(seq_along(contrast), function(i) {rbind(res_1[[i]], res_3[[i]])})
data_2 <- lapply(seq_along(contrast), function(i) {rbind(res_2[[i]], res_4[[i]])})
data_3 <- res_5

In [ ]:
data_pp <- function(res) {
    
    alpha <- 0.1
    group.by <- "cell_type_leiden_subset"
    
    # Beeswarm 
    data_1 <- suppressMessages(plotDAbeeswarm(res[[1]], group.by="cell_type_leiden_subset", alpha=alpha))$data %>% dplyr::mutate(group="Tx")
    data_2 <- suppressMessages(plotDAbeeswarm(res[[2]], group.by="cell_type_leiden_subset", alpha=alpha))$data %>% dplyr::mutate(group="D14")
    data_3 <- suppressMessages(plotDAbeeswarm(res[[3]], group.by="cell_type_leiden_subset", alpha=alpha))$data %>% dplyr::mutate(group="D100") 
    data_4 <- suppressMessages(plotDAbeeswarm(res[[4]], group.by="cell_type_leiden_subset", alpha=alpha))$data %>% dplyr::mutate(group="GVHD")
                               
    data <- rbind(data_1, data_2, data_3, data_4)
    data$group <- factor(data$group, levels=c("Tx", "D14", "D100", "GVHD"))
    
    data <- data %>%
        mutate(is_signif=ifelse(SpatialFDR < alpha, 1, 0)) %>%
        mutate(logFC_color=ifelse(is_signif==1, logFC, NA)) %>%
        arrange(group_by) %>%
        mutate(Nhood=factor(Nhood, levels=unique(Nhood))) %>%
        mutate(pos_x=pos_x, pos_y=pos_y)

    return(data)

}

In [ ]:
data_1 <- data_pp(data_1)
data_2 <- data_pp(data_2)
data_3 <- data_pp(data_3)

In [ ]:
data_1$family_domain_subset <- "Blood (CD45+)"
data_2$family_domain_subset <- "Skin (CD45+)"
data_3$family_domain_subset <- "Skin (CD45-)"

In [ ]:
data <- rbind(data_1, data_2, data_3) %>% 
        dplyr::mutate(family_domain_subset=factor(family_domain_subset, levels=c("Blood (CD45+)", "Skin (CD45+)", "Skin (CD45-)"))) %>% 
        dplyr::mutate(group_by=factor(group_by, levels=rev(names(color$cell_type_leiden_subset_r))))

In [ ]:
y_breaks <- c(-ceiling(max(abs(data$logFC))), -ceiling(max(abs(data$logFC))/2), 0, ceiling(max(abs(data$logFC))/2), ceiling(max(abs(data$logFC))))
    
dabp <- ggplot(data, aes(x=group_by, y=pos_y, color=logFC_color)) +
    scale_color_gradient2() +
    guides(color="none") +
    xlab("") + ylab("Log Fold Change") +
    scale_y_continuous(breaks=y_breaks, labels=y_breaks, limits=c(min(y_breaks), max(y_breaks))) +
    geom_hline(yintercept=0, size=0.2) + 
    geom_jitter(size=0.1, width=0.25) +
    stat_summary(aes(x=group_by, y=logFC), fun.y=median, geom="point", size=0.5, color="black") +
    facet_grid(family_domain_subset~group, space="free", scales="free_y") + 
    coord_flip() +
    scale_x_discrete(labels=setNames(parse(text=names(color$cell_type_leiden_subset_r)), names(color$cell_type_leiden_subset_r))) + 

    theme(axis.text.y=element_text(vjust=0.5, hjust=1)) 

In [ ]:
pdf("result/figures/figure_1/milor_time_point.pdf", width=4, height=6)

dabp

dev.off()

# Vireo genotype

In [ ]:
vireo <- read.csv("result/genotype/genotype_mode_1a_genome1k.csv", row.names=1) # all QC cells

In [ ]:
mat <- dplyr::filter(vireo, patient_id!="Patient_2", patient_id!="Patient_9", patient_id!="Patient_1") %>% 
    dplyr::mutate(genotype_class=factor(genotype_class, levels=names(color$genotype_class))) %>% 
    dplyr::mutate(time_point=factor(time_point, levels=names(color$time_point))) %>% 
    dplyr::mutate(cell_type_domain=replace(cell_type_domain, cell_type_domain=="non-immune cell", "CD45\u002D")) %>% 
    dplyr::mutate(cell_type_domain=replace(cell_type_domain, cell_type_domain=="immune cell", "CD45\u002B")) %>% 
    dplyr::mutate(cell_type_domain=factor(cell_type_domain, levels=c("CD45\u002D", "CD45\u002B"))) 

In [ ]:
mat <- data.frame(

    genotype_class=rep(mat$genotype_class, 2), 
    category=c(mat$time_point, mat$cell_type_domain), 
    type=factor(rep(c("Time point", "Cell type"), each=nrow(mat)), levels=c("Time point", "Cell type"))

)   

In [ ]:
p_1 <- ggplot(mat, aes(x=category, fill=genotype_class)) + 
    geom_bar(position="fill", width=0.8, color="black", size=0.1) + 
    scale_fill_manual(values=color$genotype) + 
    facet_grid(~type, scales="free", space="free") + 
    theme(axis.text.x=element_text(angle=45, vjust=1, hjust=1))  +  
    guides(fill=guide_legend(ncol=1, override.aes=list(alpha=1, size=2), keywidth=0.2, keyheight=0.2, default.unit="cm"))

In [ ]:
pdf("result/figures/figure_1/vireo_genotype_qc.pdf", width=2.5, height=2)

p_1

dev.off()

# Variant SNPs

In [ ]:
gt_vep <- read.csv("result/genotype/gt_vep.csv", row.names=1)

## Discriminate variant heatmap 

In [ ]:
mat <- dplyr::filter(gt_vep,  VARIANT, ((GT_DONOR0=="1/1" & GT_DONOR1=="0/0") | (GT_DONOR0=="0/0" & GT_DONOR1=="1/1")), PATIENT_ID!="Patient_2", PATIENT_ID!="Patient_9", PATIENT_ID!="Patient_1") %>% 
    
    # Get genotype likelihood and filter for highly confident hits 
    dplyr::mutate(LP0=sapply(strsplit(DONOR0, ":"), "[", 4), LP1=sapply(strsplit(DONOR1, ":"), "[", 4)) %>% 
    dplyr::mutate(GTLP0=ifelse(GT_DONOR0=="1/1", sapply(strsplit(LP0, ","), "[", 3), sapply(strsplit(LP0, ","), "[", 1)), GTLP1=ifelse(GT_DONOR1=="1/1", sapply(strsplit(LP1, ","), "[", 3), sapply(strsplit(LP1, ","), "[", 1))) %>% 
    dplyr::filter(GTLP0==0 & GTLP1==0) %>% 
    
    # Compute MAF 
    dplyr::mutate(MAF0=AD_DONOR0/DP_DONOR0, MAF1=AD_DONOR1/DP_DONOR1) %>% 
    
    # Make unique IDs for patients for subsetting 
    dplyr::mutate(ID=paste(SYMBOL, "|", ID)) %>% 
    
    # Select columns 
    dplyr::select(ID, PATIENT_ID, GTC_DONOR0, GTC_DONOR1, MAF0, MAF1, GT_DONOR0, GT_DONOR1) %>% 
    
    # Remove potential VEP duplicates 
    distinct %>% 

    # Set classes and factor level 
    dplyr::mutate(GTC_HOST=ifelse(GTC_DONOR0=="Host", GT_DONOR0, GT_DONOR1), GTC_DONOR=ifelse(GTC_DONOR0=="Donor", GT_DONOR0, GT_DONOR1)) %>% 
    dplyr::mutate(AD_HOST=ifelse(GTC_DONOR0=="Host", MAF0, MAF1), AD_DONOR=ifelse(GTC_DONOR0=="Donor", MAF0, MAF1)) %>% 
    dplyr::mutate(PATIENT_ID=factor(PATIENT_ID, levels=c("Patient_3", "Patient_4", "Patient_5", "Patient_6", "Patient_7", "Patient_8", "Patient_10", "Patient_11"))) %>%
    dplyr::select(ID, PATIENT_ID, AD_HOST, AD_DONOR) %>% 

    # wide to long format 
    melt(., id.vars=c("ID", "PATIENT_ID")) %>%  
    dplyr::rename(GTC=variable) %>% 
    dplyr::mutate(GTC=ifelse(GTC=="AD_HOST", "Host", "Donor")) %>% dplyr::mutate(GTC=factor(GTC, levels=c("Host", "Donor")))

In [ ]:
mat <- lapply(levels(mat$PATIENT_ID), function(PATIENT_ID) {
    
    # Get PATIENT_ID SNP IDs
    id_i <- mat[mat$PATIENT_ID==PATIENT_ID, ] %>% 
        dcast(., ID + PATIENT_ID ~ GTC, value.var="value") %>% 
        dplyr::mutate(ORDER=Host-Donor) %>%
        dplyr::arrange(-ORDER) %>% dplyr::pull(ID)
    
    # Subset mat 
    mat_i <- dcast(mat, PATIENT_ID + GTC ~ ID, value.var="value") %>% 
        replace(., is.na(.), 0)
    
    mat_i <- mat_i[, c("PATIENT_ID", "GTC", id_i)] 
    colnames(mat_i) <- c("PATIENT_ID", "GTC", paste(id_i, "|", PATIENT_ID))
    
    return(mat_i)
    
}
      )

In [ ]:
mat <- plyr::join_all(mat, by=c("PATIENT_ID", "GTC"), type='left')

In [ ]:
row_annotation_df <- dplyr::select(mat, PATIENT_ID, GTC) 

In [ ]:
# Remove categorical variables from mat
mat <- mat[, -c(1:2)]

In [ ]:
hm_list <- lapply(1:nrow(mat), function(idx, mat) {
    
    # Get idx max
    idx_max <- nrow(mat)
    
    # Subset mat and row_annotation_df
    mat <- mat[idx, , drop=FALSE] 
    
    # Get row annotation 
    row_annotation <- rowAnnotation(
        
        PATIENT_ID=row_annotation_df[idx, , drop=FALSE][, "PATIENT_ID", drop=TRUE], 
     

            annotation_legend_param=list(

                PATIENT_ID=list(title="Patient ID", title_gp=gpar(fontsize=8, fontface="plain"), labels_gp=gpar(fontsize=6), grid_width=unit(2, "mm"), grid_height=unit(2, "mm"))
            ), 

            col=list(

                PATIENT_ID=color$patient_id
            ), 

            simple_anno_size=unit(2.5, "mm"), 
            show_annotation_name=FALSE, 
            show_legend=FALSE
    
    )
    
    # Color function 
    color_ramp_mat <- c("#FFFFFF", color$genotype_class[row_annotation_df$GTC[idx]])
    breaks_mat <- seq(0, 1, length.out=2)
    color_function_mat <- circlize::colorRamp2(breaks_mat, color_ramp_mat) 
    
    hm <- ComplexHeatmap::Heatmap(
    
        matrix=mat, 
        name=paste("GTC", row_annotation_df$PATIENT_ID[idx], row_annotation_df$GTC[idx]), 

        col=color_function_mat, 
        na_col="white", 

        width=unit(0.01*ncol(mat), "mm"), 
        height=unit(2.5*nrow(mat), "mm"), 

        row_title=NULL, 
        row_title_gp=gpar(fontsize=8, fontface="bold"), 

        # column_title="DEA gene module", 
        column_title_gp=gpar(fontsize=8, fontface="bold"), 

        row_names_gp=gpar(fontsize=8, fontface="plain"), 
        column_names_gp=gpar(fontsize=8, fontface="plain"), 

        # cluster_rows=function(mat) hclust(dist(mat), method="complete"), 
        cluster_rows=FALSE, 
        show_row_dend=FALSE,   
        # row_split=row_annotation_df$PATIENT_ID, 
        row_gap=unit(0, "mm"), 
        show_row_names=FALSE, 
        row_names_side="right", 

        cluster_columns=FALSE, 
        cluster_column_slices=FALSE, 
        show_column_dend=TRUE, 
        # column_split=column_split, 
        column_gap=unit(2, "mm"), 
        show_column_names=FALSE, 

        top_annotation=NULL, 
        left_annotation=row_annotation, 

        rect_gp=gpar(col=NA, lwd=0, alpha=1), 
        
        heatmap_legend_param=list(title=paste0("MAF (", row_annotation_df$GTC[idx], ")"), title_gp=gpar(fontsize=6, fontface="plain"), labels_gp=gpar(fontsize=6), grid_height=unit(2, "mm"), legend_width=unit(10, "mm"), direction="horizontal"),
        show_heatmap_legend=ifelse(idx<3, TRUE, FALSE), 

        border=FALSE, 
        border_gp=gpar(col="black", lwd=unit(0.5, "mm")), 

        use_raster=FALSE, raster_by_magick=FALSE

    )
    
    return(hm)
    
    
}, mat=mat
            )

In [ ]:
hm <- Reduce(`%v%`, hm_list)

In [ ]:
pdf("result/figures/figure_1/maf_discriminative_snp.pdf", width=10, height=6)

draw(hm, gap=unit(0, "mm"), column_title="MAF of discriminative SNPs", column_title_gp=gpar(fontsize=8, fontface="bold"), heatmap_legend_side="bottom") 

dev.off()

## Gene expression level of SNPs 

In [ ]:
mat <- dplyr::filter(gt_vep,  VARIANT, ((GT_DONOR0=="1/1" & GT_DONOR1=="0/0") | (GT_DONOR0=="0/0" & GT_DONOR1=="1/1")), PATIENT_ID!="Patient_2", PATIENT_ID!="Patient_9", PATIENT_ID!="Patient_1") %>% 
    
    # Get genotype likelihood and filter for highly confident hits 
    dplyr::mutate(LP0=sapply(strsplit(DONOR0, ":"), "[", 4), LP1=sapply(strsplit(DONOR1, ":"), "[", 4)) %>% 
    dplyr::mutate(GTLP0=ifelse(GT_DONOR0=="1/1", sapply(strsplit(LP0, ","), "[", 3), sapply(strsplit(LP0, ","), "[", 1)), GTLP1=ifelse(GT_DONOR1=="1/1", sapply(strsplit(LP1, ","), "[", 3), sapply(strsplit(LP1, ","), "[", 1))) %>% 
    dplyr::filter(GTLP0==0 & GTLP1==0) %>% 

    # Subset data 
    dplyr::select(PATIENT_ID, SYMBOL, PCT) %>% 
    na.omit() %>% distinct %>% 
    dplyr::group_by(SYMBOL, PCT) %>% dplyr::summarise(SYMBOL_N=n()) %>% 
    dplyr::arrange(-PCT) %>% tibble::rowid_to_column("RANK") %>% 
    dplyr::mutate(LABEL=ifelse(RANK<=20, SYMBOL, NA))

In [ ]:
dp <- ggplot(mat, aes(x=PCT, y=RANK, size=SYMBOL_N, label=LABEL)) + 
    geom_point(shape=21, fill="dark gray", color="dark gray", stroke=0.05) + 
    ggtitle("SNP ranking") + xlab("Cells expressing [%]") + ylab("log(Rank)") +
    xlim(0, 1) + 
    scale_y_log10() + 
    # scale_x_reverse() + scale_y_reverse() + 
    geom_text_repel(segment.color="black", force=10, force_pull=1, max.overlaps=getOption("ggrepel.max.overlaps", default=100), size=2, alpha=1, guide="none", segment.size=0.05, color="black", min.segment.length=0) + 
    scale_size(range=c(1, 3)) + 
    theme_global_set(4) + coord_flip()

In [ ]:
pdf("result/figures/figure_1/discriminative_snp_ranking.pdf", width=3, height=2.5)

dp

dev.off()

## Variance effect prediction analysis (VEP)

In [ ]:
mat <- dplyr::filter(gt_vep,  VARIANT, ((GT_DONOR0=="1/1" & GT_DONOR1=="0/0") | (GT_DONOR0=="0/0" & GT_DONOR1=="1/1")), PATIENT_ID!="Patient_2", PATIENT_ID!="Patient_9", PATIENT_ID!="Patient_1") %>% 
    
    # Get genotype likelihood and filter for highly confident hits 
    dplyr::mutate(LP0=sapply(strsplit(DONOR0, ":"), "[", 4), LP1=sapply(strsplit(DONOR1, ":"), "[", 4)) %>% 
    dplyr::mutate(GTLP0=ifelse(GT_DONOR0=="1/1", sapply(strsplit(LP0, ","), "[", 3), sapply(strsplit(LP0, ","), "[", 1)), GTLP1=ifelse(GT_DONOR1=="1/1", sapply(strsplit(LP1, ","), "[", 3), sapply(strsplit(LP1, ","), "[", 1))) %>% 
    dplyr::filter(GTLP0==0 & GTLP1==0) %>% 

    # Prepare matrix
    dplyr::select(ID, PATIENT_ID, GVHD_CLASS, CONSEQUENCE, BAM_READS_ALIGNED) %>% 
    na.omit() %>% dplyr::distinct() 

In [ ]:
chi_square_data <- dplyr::group_by(mat, GVHD_CLASS, CONSEQUENCE) %>% 
    dplyr::summarise(N=n()) %>% 
    dcast(., CONSEQUENCE~GVHD_CLASS, value.var="N") %>% 
    na.omit() %>% dplyr::filter(GVHD_dev>=5, GVHD_free>=5) %>% 
    tibble::column_to_rownames("CONSEQUENCE") %>% dplyr::select(GVHD_free, GVHD_dev)

In [ ]:
# Chi square test and standardized residuals
chi_square_test <- chisq.test(chi_square_data)
std_residuals <- chi_square_test$stdres %>% as.data.frame()

In [ ]:
reactable(
    
    tibble::rownames_to_column(std_residuals, "CONSEQUENCE") %>% mutate_if(is.numeric, round, 4) %>% dplyr::mutate(CONSEQUENCE=gsub("_|,", " ", CONSEQUENCE)),
    
    striped=TRUE,
    
    defaultColDef=colDef(
        align="left"
    ),
    columns=list(
        CONSEQUENCE=colDef(minWidth=200, filterable=TRUE),
        p_value=colDef(width=100),
        p_adj=colDef(width=100)
    ),
    style=list(maxWidth=700), 
    
    theme=reactableTheme(
        style=list(fontFamily="Arial")
    ), 
    
    outlined=TRUE, 
    filterable=FALSE, 
    defaultPageSize=nrow(std_residuals), 
)

In [ ]:
# Fit GLMM
glmm_data <- dplyr::group_by(mat, PATIENT_ID, GVHD_CLASS, CONSEQUENCE, BAM_READS_ALIGNED) %>% 
    dplyr::summarise(N=n()) %>% 
    dplyr::filter(CONSEQUENCE %in% rownames(chi_square_data))

In [ ]:
# Set factor levels
glmm_data$GVHD_CLASS <- relevel(factor(glmm_data$GVHD_CLASS), ref="GVHD_free")
glmm_data$CONSEQUENCE <- relevel(factor(glmm_data$CONSEQUENCE), ref="upstream_gene_variant")

In [ ]:
# Fit the GLMM
glmm_model <- glmer.nb(N ~ GVHD_CLASS * CONSEQUENCE + (1 | PATIENT_ID) + offset(log(BAM_READS_ALIGNED)) , data=glmm_data)

In [ ]:
interaction_terms <- summary(glmm_model)$coefficients[grepl("GVHD_CLASSGVHD_dev:CONSEQUENCE", rownames(summary(glmm_model)$coefficients)), c("Estimate", "Pr(>|z|)")] %>% as.data.frame() 
colnames(interaction_terms) <- c("Estimate", "p_value")
interaction_terms$CONSEQUENCE <- gsub("GVHD_CLASSGVHD_dev:CONSEQUENCE", "", rownames(interaction_terms))
rownames(interaction_terms) <- 1:nrow(interaction_terms)
interaction_terms <- interaction_terms[order(interaction_terms$p_value), ]
interaction_terms$p_adj <- p.adjust(interaction_terms$p_value, method="BH")

In [ ]:
reactable(
    
    interaction_terms[, c("CONSEQUENCE", "Estimate", "p_value", "p_adj")] %>% mutate_if(is.numeric, round, 2) %>% dplyr::mutate(CONSEQUENCE=gsub("_|,", " ", CONSEQUENCE)),
    
    striped=TRUE,
    
    defaultColDef=colDef(
        align="left"
    ),
    columns=list(
        CONSEQUENCE=colDef(minWidth=200, filterable=TRUE),
        Estimate=colDef(width=100),
        p_value=colDef(width=100),
        p_adj=colDef(width=100)
    ),
    style=list(maxWidth=700), 
    
    theme=reactableTheme(
        style=list(fontFamily="Arial")
    ), 
    
    outlined=TRUE, 
    filterable=FALSE, 
    defaultPageSize=nrow(interaction_terms), 
)

In [ ]:
dp <- mat %>% 
    dplyr::mutate(CONSEQUENCE=ifelse(CONSEQUENCE %in% interaction_terms[interaction_terms$p_adj<=0.05, ]$CONSEQUENCE, CONSEQUENCE, "Other")) %>%
    dplyr::group_by(GVHD_CLASS, CONSEQUENCE) %>% 
    dplyr::summarise(N=n()) %>% 
    dplyr::group_by(GVHD_CLASS) %>% dplyr::mutate(N_TOTAL=sum(N)) %>% dplyr::ungroup() %>% 
    dplyr::mutate(RATIO=N/N_TOTAL) %>% 
    dplyr::mutate(GVHD_CLASS=factor(GVHD_CLASS, levels=c("GVHD_free", "GVHD_dev"))) %>%
    dplyr::mutate(CONSEQUENCE=factor(CONSEQUENCE, levels=c("missense_variant", "synonymous_variant", "3_prime_UTR_variant", "Other"))) %>% 

    ggplot(., aes(x=2, y=RATIO, fill=CONSEQUENCE)) +
        geom_bar(stat="identity", color="white") +
        coord_polar(theta="y", start=0) + 
        scale_fill_brewer(palette="Blues") + 
        facet_grid(~GVHD_CLASS) +
        xlim(0.5, 2.5) + 
        theme_global_set(4)
    

In [ ]:
pdf("result/figures/figure_1/vep_circular.pdf", width=4, height=4)

dp

dev.off()